In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack, csr_matrix

df = pd.read_csv('../data/interview_cleaned_for_training.csv')
print("Shape ya awali:", df.shape)

Shape ya awali: (2000000, 7)


In [2]:
df['Gaze_Extracted'] = df['Eye_Movement_Telemetry'].str.extract(r'Max Gaze Deviation=([\d.]+)°').astype(float)
df['OffScreen_Extracted'] = df['Eye_Movement_Telemetry'].str.extract(r'Off-screen Cumulative Time=([\d.]+)s').astype(float)

print("Columns baada ya extraction:", df.columns.tolist())
print()
print("Gaze_Extracted unique count:", df['Gaze_Extracted'].nunique())
print("OffScreen_Extracted unique count:", df['OffScreen_Extracted'].nunique())

Columns baada ya extraction: ['Question_ID', 'Academic_Field', 'User_Specialization', 'Targeted_User_Prompt', 'Humanlike_AI_Response', 'Eye_Movement_Telemetry', 'Proctoring_Cheating_Label', 'Gaze_Extracted', 'OffScreen_Extracted']

Gaze_Extracted unique count: 7
OffScreen_Extracted unique count: 7


In [3]:
train_idx, test_idx = train_test_split(
    df.index, 
    test_size=0.3, 
    stratify=df['Proctoring_Cheating_Label'], 
    random_state=42
)

train_df = df.loc[train_idx].copy()
test_df = df.loc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()
print("Train columns:", train_df.columns.tolist())

Train shape: (1400000, 9)
Test shape: (600000, 9)

Train columns: ['Question_ID', 'Academic_Field', 'User_Specialization', 'Targeted_User_Prompt', 'Humanlike_AI_Response', 'Eye_Movement_Telemetry', 'Proctoring_Cheating_Label', 'Gaze_Extracted', 'OffScreen_Extracted']


In [4]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

train_cat_encoded = ohe.fit_transform(train_df[['Academic_Field', 'User_Specialization']])
test_cat_encoded = ohe.transform(test_df[['Academic_Field', 'User_Specialization']])

print("Category features shape (train):", train_cat_encoded.shape)
print("Category features shape (test):", test_cat_encoded.shape)
print()
print("Majina ya columns:", ohe.get_feature_names_out())

Category features shape (train): (1400000, 14)
Category features shape (test): (600000, 14)

Majina ya columns: ['Academic_Field_Business & Finance' 'Academic_Field_Computer Science'
 'Academic_Field_Engineering' 'Academic_Field_Health Sciences'
 'Academic_Field_Human Resources' 'User_Specialization_Civil Engineering'
 'User_Specialization_Corporate Finance'
 'User_Specialization_Data Science & AI'
 'User_Specialization_Digital Marketing'
 'User_Specialization_Electrical Engineering'
 'User_Specialization_Medicine & Surgery' 'User_Specialization_Nursing'
 'User_Specialization_Software Engineering'
 'User_Specialization_Talent Acquisition']


In [5]:
tfidf_prompt = TfidfVectorizer(max_features=500, stop_words='english')
train_prompt_tfidf = tfidf_prompt.fit_transform(train_df['Targeted_User_Prompt'])
test_prompt_tfidf = tfidf_prompt.transform(test_df['Targeted_User_Prompt'])

tfidf_response = TfidfVectorizer(max_features=500, stop_words='english')
train_response_tfidf = tfidf_response.fit_transform(train_df['Humanlike_AI_Response'])
test_response_tfidf = tfidf_response.transform(test_df['Humanlike_AI_Response'])

print("Prompt TF-IDF shape (train):", train_prompt_tfidf.shape)
print("Response TF-IDF shape (train):", train_response_tfidf.shape)

Prompt TF-IDF shape (train): (1400000, 62)
Response TF-IDF shape (train): (1400000, 97)


In [6]:
train_numeric_leak = train_df[['Gaze_Extracted', 'OffScreen_Extracted']].values
test_numeric_leak = test_df[['Gaze_Extracted', 'OffScreen_Extracted']].values

X_train_model1 = hstack([
    train_cat_encoded, 
    train_prompt_tfidf, 
    train_response_tfidf,
    csr_matrix(train_numeric_leak)
])

X_test_model1 = hstack([
    test_cat_encoded, 
    test_prompt_tfidf, 
    test_response_tfidf,
    csr_matrix(test_numeric_leak)
])

y_train = train_df['Proctoring_Cheating_Label']
y_test = test_df['Proctoring_Cheating_Label']

print("X_train_model1 shape:", X_train_model1.shape)
print("X_test_model1 shape:", X_test_model1.shape)

X_train_model1 shape: (1400000, 175)
X_test_model1 shape: (600000, 175)


In [7]:
model1 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model1.fit(X_train_model1, y_train)

print("Model 1 imemaliza kufundishwa!")

Model 1 imemaliza kufundishwa!


In [9]:
y_pred1 = model1.predict(X_test_model1)

print("MODEL 1 — Matokeo:")
print("Accuracy:", accuracy_score(y_test, y_pred1))
print()
print(classification_report(y_test, y_pred1))

MODEL 1 — Matokeo:
Accuracy: 1.0

                    precision    recall  f1-score   support

            Normal       1.00      1.00      1.00    300021
Suspected Cheating       1.00      1.00      1.00    299979

          accuracy                           1.00    600000
         macro avg       1.00      1.00      1.00    600000
      weighted avg       1.00      1.00      1.00    600000



UCHUNGUZI WA MWANZO UMEKAMILIKA 

In [10]:
X_train_model2 = hstack([
    train_cat_encoded, 
    train_prompt_tfidf, 
    train_response_tfidf
])

X_test_model2 = hstack([
    test_cat_encoded, 
    test_prompt_tfidf, 
    test_response_tfidf
])

print("X_train_model2 shape:", X_train_model2.shape)
print("X_test_model2 shape:", X_test_model2.shape)

X_train_model2 shape: (1400000, 173)
X_test_model2 shape: (600000, 173)


In [11]:
model2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model2.fit(X_train_model2, y_train)

print("Model 2 imemaliza kufundishwa!")

Model 2 imemaliza kufundishwa!


In [13]:
y_pred2 = model2.predict(X_test_model2)

print("MODEL 2 — Matokeo:")
print("Accuracy:", accuracy_score(y_test, y_pred2))
print()
print(classification_report(y_test, y_pred2))

MODEL 2 — Matokeo:
Accuracy: 0.4991083333333333

                    precision    recall  f1-score   support

            Normal       0.50      0.47      0.48    300021
Suspected Cheating       0.50      0.53      0.51    299979

          accuracy                           0.50    600000
         macro avg       0.50      0.50      0.50    600000
      weighted avg       0.50      0.50      0.50    600000

